### config

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import config.ConnectionConfig as cc
cc.setupEnvironment()

spark = cc.startLocalCluster("fact_rides",7)
spark.getActiveSession()

25/04/02 12:45:51 WARN Utils: Your hostname, 4L3KS-comp resolves to a loopback address: 127.0.1.1; using 10.140.98.193 instead (on interface wlp2s0)
25/04/02 12:45:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/aleks/Downloads/bigtools/spark-3.5.4-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/aleks/.ivy2/cache
The jars for the packages stored in: /home/aleks/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
org.elasticsearch#elasticsearch-spark-30_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2bf3020a-9de9-467e-85b5-19d4c5de9fca;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.

# EXTRACT

I tried using the spark API first, but there are some limitations on their joining of columns.

In [38]:
# EXTRACT rides:
# rides_table_SQL = '(SELECT * FROM rides) as rides_table'
# df_rides = spark.read.format("jdbc")\
#     .option("driver" , cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", rides_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "rideid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_rides.printSchema()
#df_rides.show()
#df_rides.count()
# ----

#EXTRACT bike_type through bike_lot through vehicle:
# Rename to avoid duplicate column names
# Corrected column name
# vehicle_table_SQL = """
# (
#     SELECT
#         v.*,
#         l.bikelotid AS bikelotid_bikelots,
#         l.deliverydate,
#         l.biketypeid
#     FROM vehicles v
#     LEFT JOIN bikelots l ON v.bikelotid = l.bikelotid
# ) AS vehicle_table
# """
#
# df_vehicles = spark.read.format("jdbc")\
#     .option("driver", cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", vehicle_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "vehicleid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_vehicles.show()
# ------

#EXTRACT users through subscriptionid through subscription
# user_table_SQL = """
# (
#     SELECT
#         userid AS userid_subscriptions,
#         subscriptionid AS subscriptionid_subscriptions
#     FROM subscriptions s
# ) AS user_table
# """
#
# df_users = spark.read.format("jdbc")\
#     .option("driver", cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", user_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "userid_subscriptions") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_users.show()

# ------

#EXTRACT date
# df_dates = spark.read.format("delta").load("../dimensions/spark-warehouse/dimdate")
# df_dates.show()
# df_dates.createOrReplaceTempView("dates")

# ------

#EXTRACT weather: TBD.



Trying to do it in one SQL query:

In [3]:
# EXTRACTING ALL IN ONE GO:
# load dates from deltatable:
df_dates = spark.read.format("delta").load("../dimensions/spark-warehouse/dimdate")
#df_dates.show()
df_dates.createOrReplaceTempView("dates")

# write the SQL string needed:
SQL = """(
    SELECT
    r.*,
    v.vehicleid AS vehicleid_vehicles,
    v.bikelotid AS bikelotid_vehicles,
    b.bikelotid AS bikelotid_bikelots,
    b.biketypeid AS biketypeid_bikelots,
    s.subscriptionid AS subscriptionid_subscriptions,
    s.userid AS userid_subscriptions
        FROM rides r
            LEFT JOIN vehicles v ON r.vehicleid = v.vehicleid
            LEFT JOIN bikelots b ON v.bikelotid = b.bikelotid
            LEFT JOIN subscriptions s ON r.subscriptionid = s.subscriptionid

) as rides_table
"""

df_rides_full = spark.read.format("jdbc")\
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", SQL) \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0)\
    .option("upperBound", 100) \
    .load()

df_rides_full.show()
# WEATHER would be done separately


+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+------------------+------------------+------------------+-------------------+----------------------------+--------------------+
|rideid|       startpoint|         endpoint|          starttime|            endtime|vehicleid|subscriptionid|startlockid|endlockid|vehicleid_vehicles|bikelotid_vehicles|bikelotid_bikelots|biketypeid_bikelots|subscriptionid_subscriptions|userid_subscriptions|
+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+------------------+------------------+------------------+-------------------+----------------------------+--------------------+
|     1|(51.2083,4.44595)|(51.1938,4.40228)|2015-09-22 00:00:00|2012-09-22 00:00:00|      844|         13296|       4849|     3188|               844|                 3|                 3|                  1|               

In [10]:
#EXTRACT
df_users = spark.read.load('./spark-warehouse/dim_user/dim_users.snappy.parquet')
df_users.createOrReplaceTempView("users_dim")
df_dates = spark.read.load('./spark-warehouse/dimdate/dim_date.snappy.parquet')
df_dates.createOrReplaceTempView("dates_dim")
df_users.show()

+-------+-------+--------------------+-------+-------+--------------------+------------+----------+----------+
|user_sk|user_id|              street| number|zipcode|                city|country_code|start_date|  end_date|
+-------+-------+--------------------+-------+-------+--------------------+------------+----------+----------+
|      0|     12|         Bergenhoeve|81 0302|   2040|Antwerpen/Berendr...|          BE|2023-10-20|2024-10-19|
|      1|     12|         Bergenhoeve|81 0302|   2040|Antwerpen/Berendr...|          BE|2021-09-09|2022-09-09|
|      2|     12|         Bergenhoeve|81 0302|   2040|Antwerpen/Berendr...|          BE|2020-08-07|2021-08-07|
|      3|     13|  Trompetvogelstraat|     1 |   2170| Merksem (Antwerpen)|          BE|2020-02-28|2021-02-27|
|      4|     14|Prosper De Vochtlaan|   177 |   2550|    Kontich/Waarloos|          BE|2023-09-19|2024-09-18|
|      5|     18|Emiel Van Hemeldo...|    93 |   2540|                Hove|          BE|2023-08-04|2024-08-03|
|

# TRANSFORM

In [13]:
#TRANSFORMATION
date_joined_df = df_dates.join(df_rides_full, df_rides_full.starttime==df_dates.CalendarDate, "left") \
    .select(
    df_rides_full.rideid,
    df_dates.dateSK.alias("dates_sk"),
    df_rides_full.startlockid.alias("start_lock_id"),
    df_rides_full.endlockid.alias("end_lock_id"),
    df_rides_full.vehicleid.alias("vehicle_id")) \
    .join()

# dataframe_rides = df_rides_full \
#     .withColumnRenamed() \ # Add more

+------+--------+-------------+-----------+----------+
|rideid|dates_sk|start_lock_id|end_lock_id|vehicle_id|
+------+--------+-------------+-----------+----------+
|  NULL|       0|         NULL|       NULL|      NULL|
|  5074|       2|         4447|       NULL|       370|
|  5075|       2|         4448|        254|       774|
|  5076|       2|         4449|        255|      5722|
|  5077|       2|         4450|       NULL|      1266|
|  5078|       2|         4451|        257|      1981|
|  5079|       2|         4452|       NULL|      2791|
|  5080|       2|         4454|        259|      5711|
|  5081|       2|         4455|        260|      5395|
| 68009|      20|         1408|        632|       141|
| 68010|      20|         1411|        633|      1526|
| 68011|      20|         1412|        634|      1019|
| 68012|      20|         1413|        636|      4182|
| 68013|      20|         1414|        637|      5686|
| 14911|       4|         6266|         55|       554|
| 14912|  

# LOAD

+-------+-------+--------------------+-------+-------+--------------------+------------+----------+----------+
|user_sk|user_id|              street| number|zipcode|                city|country_code|start_date|  end_date|
+-------+-------+--------------------+-------+-------+--------------------+------------+----------+----------+
|      0|     12|         Bergenhoeve|81 0302|   2040|Antwerpen/Berendr...|          BE|2023-10-20|2024-10-19|
|      1|     12|         Bergenhoeve|81 0302|   2040|Antwerpen/Berendr...|          BE|2021-09-09|2022-09-09|
|      2|     12|         Bergenhoeve|81 0302|   2040|Antwerpen/Berendr...|          BE|2020-08-07|2021-08-07|
|      3|     13|  Trompetvogelstraat|     1 |   2170| Merksem (Antwerpen)|          BE|2020-02-28|2021-02-27|
|      4|     14|Prosper De Vochtlaan|   177 |   2550|    Kontich/Waarloos|          BE|2023-09-19|2024-09-18|
|      5|     18|Emiel Van Hemeldo...|    93 |   2540|                Hove|          BE|2023-08-04|2024-08-03|
|

In [ ]:
spark.stop()